In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.data_processing import (
    extract_condition, 
    extract_day,
    classify_ki67_status
)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

In [ ]:
# Notebook runs from notebooks/, so results is one level up
results_dir = Path("../ki67_results").resolve()
output_dir = results_dir / "processed_csvs"
output_file = output_dir / "processed_ki67.csv"

# Find every CSV under results/ recursively, excluding the final output file if it already exists
csv_files = sorted(
    csv_path
    for csv_path in results_dir.rglob("*.csv")
    if csv_path.resolve() != output_file.resolve()
)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {results_dir}")

# Read and merge all CSV files
merged_df = pd.concat((pd.read_csv(csv_file) for csv_file in csv_files), ignore_index=True)

# Extract 'condition' and 'day' metadata columns and store them
merged_df["condition"] = merged_df["lif_container_id"].apply(extract_condition)
merged_df["day"] = merged_df["lif_image_name"].apply(extract_day)

# Ensure output directory exists and save merged CSV
output_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(output_file, index=False)

print(f"Found results directory: {results_dir}")
print(f"Merged {len(csv_files)} CSV files into: {output_file}")
print(f"Merged dataframe shape: {merged_df.shape}")
merged_df.head()

In [ ]:
# Filter out rows where area is below 500 or above 6500
filtered_df = merged_df[(merged_df["area"] >= 500) & (merged_df["area"] <= 6500)]

# Percentile-based filtering for ki67_mean_int (between 1st and 99th percentiles)
ki67_lower = np.percentile(filtered_df["ki67_mean_int"], 1)
ki67_upper = np.percentile(filtered_df["ki67_mean_int"], 99)

filtered_df = filtered_df[
    (filtered_df["ki67_mean_int"] >= ki67_lower) &
    (filtered_df["ki67_mean_int"] <= ki67_upper)
]

# Replace merged_df with filtered_df so all downstream cells use the filtered data
merged_df = filtered_df

In [ ]:
# Loop over days 3, 7, and 14 and produce the strip plots for each day
for day in [3, 7, 14]:
    df_day = merged_df[merged_df["day"] == day]

    plt.figure(figsize=(6, 5))
    ax = sns.stripplot(
        data=df_day,
        x="condition",
        y="ki67_mean_int",
        hue="condition",
        palette={"static": "red", "perfusion": "blue"},
        dodge=False
    )
    ax.set_title(f"Day {day} Ki67 Mean Intensity by Condition")
    ax.set_xlabel("Condition")
    ax.set_ylabel("Ki67 Mean Intensity")
    plt.show()

In [ ]:
# Loop over days 3, 7, and 14 and produce the violin plots for each day
for day in [3, 7, 14]:
    df_day = merged_df[merged_df["day"] == day]

    plt.figure(figsize=(6, 5))
    ax = sns.violinplot(
        data=df_day,
        x="condition",
        y="ki67_mean_int",
        hue="condition",
        palette={"static": "red", "perfusion": "blue"},
        legend=False,
        dodge=False
    )
    ax.set_title(f"Day {day} Ki67 Mean Intensity by Condition")
    ax.set_xlabel("Condition")
    ax.set_ylabel("Ki67 Mean Intensity")
    plt.show()

In [ ]:
# Plot violin plots for days 3, 7, and 14 side by side, grouping by day
plt.figure(figsize=(10, 6))
ax = sns.violinplot(
    data=merged_df[merged_df["day"].isin([3, 7, 14])],
    x="day",
    y="ki67_mean_int",
    hue="condition",
    palette={"static": "red", "perfusion": "blue"},
    dodge=True,
    cut=0
)
ax.set_title("Ki67 Mean Intensity by Day and Condition")
ax.set_xlabel("Day")
ax.set_ylabel("Ki67 Mean Intensity")
plt.legend(title="Condition")
plt.show()

In [ ]:
ki67_df = classify_ki67_status(merged_df, feature_columns=['ki67_mean_int'])

In [ ]:
# Calculate percentage of Ki67 positive cells per lif_image_name, also including condition and day.
ki67_perc_df = (
    ki67_df.groupby(['lif_image_name', 'condition', 'day'])['ki67_status']
    .agg(
        n_cells='count',
        n_ki67_pos='sum'
    )
    .reset_index()
)
ki67_perc_df['ki67_pos_pct'] = 100 * ki67_perc_df['n_ki67_pos'] / ki67_perc_df['n_cells']

# Plot with Plotly for interactive hover (lif_image_name visible)
fig = px.strip(
    ki67_perc_df,
    x="day",
    y="ki67_pos_pct",
    color="condition",
    hover_data=["lif_image_name", "n_cells", "n_ki67_pos"],
    stripmode="overlay",
)
fig.update_traces(jitter=0.4, marker=dict(size=9, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    yaxis_title='Percentage of Ki67 Positive Cells',
    xaxis_title='Day',
    title='Percentage of Ki67+ Cells per Image',
    width=1200,
    height=600,
    xaxis_tickangle=-45
)
fig.show()